# 🚀 DiabetAI v10 - Massive Synthetic Injection

**Problem:** Correction Bolus Blindness - Model ignores high BG corrections

**Solution:** Inject 2,000,000 synthetic correction examples
- CGM: 140-450 mg/dL (Focus on HIGH)
- Carbs: STRICTLY 0
- Target: Clinical formula `(CGM-100)/ISF - IOB`

**Strategy:**
- 2M synthetic rows (Weight 2.0x = 4M effective)
- ~17.5M real rows (Weight 1.0x)
- Signal-to-Noise Ratio: ~10%

In [ ]:
!pip install polars xgboost matplotlib scikit-learn -U -q

import polars as pl
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import json
import os
import gc

from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ XGBoost: {xgb.__version__}")
print(f"✅ Polars: {pl.__version__}")

# GPU Check
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], 
                       capture_output=True, text=True)
print(f"\n🖥️ GPU:\n{result.stdout}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
MODEL_PATH = "/content/drive/MyDrive/DiabetesKDS/models"

TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_balanced_v7.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_real_world_v7.parquet"

TARGET = 'bolus'
TARGET_BG = 100.0  # Target glucose for correction calculation

# Synthetic injection size
SYNTHETIC_SIZE = 2_000_000  # 2 Million

print(f"📂 Train: {TRAIN_FILE}")
print(f"📂 Test: {TEST_FILE}")
print(f"\n🧬 Synthetic Injection: {SYNTHETIC_SIZE:,} rows")

## Step 1: Generate 2,000,000 Synthetic Correction Rows

In [ ]:
print("="*70)
print("🧬 GENERATING MASSIVE SYNTHETIC DATA")
print("="*70)

np.random.seed(42)

def generate_massive_synthetic_data(n=2_000_000):
    """
    Generate 2M synthetic CORRECTION examples.
    Key: carbs=0, high CGM, variety of ISF/IOB combinations.
    """
    print(f"\n🔄 Generating {n:,} synthetic correction examples...")
    
    # CGM: Focus on HIGH values (140-450)
    # Bias towards higher values
    cgm_low = np.random.randint(140, 200, size=n // 3)
    cgm_mid = np.random.randint(200, 300, size=n // 3)
    cgm_high = np.random.randint(300, 451, size=n - 2*(n//3))
    cgm = np.concatenate([cgm_low, cgm_mid, cgm_high]).astype(np.float32)
    np.random.shuffle(cgm)
    
    # Carbs: STRICTLY 0 (This is the critical lesson)
    carbs = np.zeros(n, dtype=np.float32)
    
    # ISF: Cover all common sensitivities
    isf_options = np.array([20, 25, 30, 35, 40, 45, 50, 60], dtype=np.float32)
    est_isf = np.random.choice(isf_options, size=n)
    
    # ICR: Typical range
    icr_options = np.array([8, 10, 12, 15, 18, 20], dtype=np.float32)
    est_icr = np.random.choice(icr_options, size=n)
    
    # IOB: 0 to 4 units (teach IOB braking)
    iob = np.random.uniform(0.0, 4.0, size=n).astype(np.float32)
    
    # TDD: Estimate based on ISF
    est_tdd = (1800.0 / est_isf).astype(np.float32)
    
    # Time features: Random distribution
    hours = np.random.uniform(0, 24, size=n)
    time_sin = np.sin(2 * np.pi * hours / 24).astype(np.float32)
    time_cos = np.cos(2 * np.pi * hours / 24).astype(np.float32)
    is_weekend = np.random.choice([0.0, 1.0], size=n, p=[0.7, 0.3]).astype(np.float32)
    
    # Trends: Slight positive bias for high BG
    cgm_delta_5min = np.random.uniform(-2, 5, size=n).astype(np.float32)
    cgm_delta_15min = (cgm_delta_5min * 3).astype(np.float32)
    cgm_velocity = (cgm_delta_15min / 15).astype(np.float32)
    
    # Basal
    basal = np.random.uniform(0.5, 2.0, size=n).astype(np.float32)
    basal_delta_5min = np.zeros(n, dtype=np.float32)
    
    # Demographics
    age = np.random.randint(18, 70, size=n).astype(np.float32)
    weight = np.random.randint(50, 120, size=n).astype(np.float32)
    gender_encoded = np.random.choice([0.0, 1.0], size=n).astype(np.float32)
    
    # ========== PHYSICS FEATURES ==========
    # Meal bolus: 0 (no carbs)
    phys_meal_bolus = np.zeros(n, dtype=np.float32)
    
    # Correction bolus: (CGM - target) / ISF
    phys_corr_bolus = np.maximum(0, (cgm - TARGET_BG) / est_isf).astype(np.float32)
    
    # Net need: correction - IOB
    phys_net_need = np.maximum(0, phys_corr_bolus - iob).astype(np.float32)
    
    # ========== TARGET LABEL ==========
    # THE GOLDEN CLINICAL RULE: (CGM - 100) / ISF - IOB
    bolus = np.maximum(0, (cgm - TARGET_BG) / est_isf - iob).astype(np.float32)
    
    # Create DataFrame
    df = pl.DataFrame({
        'CGM': cgm,
        'carbs': carbs,
        'iob': iob,
        'phys_meal_bolus': phys_meal_bolus,
        'phys_corr_bolus': phys_corr_bolus,
        'phys_net_need': phys_net_need,
        'est_tdd': est_tdd,
        'est_isf': est_isf,
        'est_icr': est_icr,
        'cgm_delta_5min': cgm_delta_5min,
        'cgm_delta_15min': cgm_delta_15min,
        'cgm_velocity': cgm_velocity,
        'basal': basal,
        'basal_delta_5min': basal_delta_5min,
        'time_sin': time_sin,
        'time_cos': time_cos,
        'is_weekend': is_weekend,
        'age': age,
        'weight': weight,
        'gender_encoded': gender_encoded,
        'bolus': bolus,
        'is_synthetic': np.ones(n, dtype=np.float32)  # Tag for weighting
    })
    
    return df

# Generate!
df_synthetic = generate_massive_synthetic_data(n=SYNTHETIC_SIZE)

print(f"\n✅ Synthetic data generated: {len(df_synthetic):,} rows")
print(f"\n📊 Synthetic CGM Distribution:")
print(df_synthetic.select('CGM').describe())
print(f"\n📊 Synthetic Bolus Distribution:")
print(df_synthetic.select('bolus').describe())

In [ ]:
# Visualize synthetic data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CGM Distribution
axes[0].hist(df_synthetic.select('CGM').to_numpy(), bins=50, color='#ff4b4b', alpha=0.7)
axes[0].set_xlabel('CGM (mg/dL)')
axes[0].set_title('Synthetic CGM Distribution\n(Focus on HIGH)')
axes[0].axvline(x=180, color='orange', linestyle='--', label='High BG Threshold')
axes[0].legend()

# Bolus Distribution
axes[1].hist(df_synthetic.select('bolus').to_numpy(), bins=50, color='#667eea', alpha=0.7)
axes[1].set_xlabel('Bolus (U)')
axes[1].set_title('Synthetic Bolus Distribution')

# CGM vs Bolus
sample = df_synthetic.sample(n=10000)
axes[2].scatter(sample.select('CGM').to_numpy(), sample.select('bolus').to_numpy(), 
                alpha=0.3, s=1, c='#38ef7d')
axes[2].set_xlabel('CGM (mg/dL)')
axes[2].set_ylabel('Bolus (U)')
axes[2].set_title('CGM vs Correction Bolus\n(The Pattern We Want)')

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/v10_synthetic_distribution.png", dpi=150)
plt.show()

## Step 2: Load Real Data & Merge

In [ ]:
print("="*70)
print("📂 LOADING & MERGING DATA")
print("="*70)

# Load real data
print("\n📂 Loading real training data...")
df_real = pl.read_parquet(TRAIN_FILE)
print(f"✅ Real data: {len(df_real):,} rows")

# Add physics features to real data
def add_physics_features(df):
    return df.with_columns([
        (pl.col('carbs') / pl.col('est_icr').clip(lower_bound=1.0))
            .fill_nan(0.0).fill_null(0.0)
            .alias('phys_meal_bolus'),
        ((pl.col('CGM') - TARGET_BG) / pl.col('est_isf').clip(lower_bound=1.0))
            .clip(lower_bound=0.0)
            .fill_nan(0.0).fill_null(0.0)
            .alias('phys_corr_bolus'),
    ]).with_columns([
        (pl.col('phys_meal_bolus') + pl.col('phys_corr_bolus') - pl.col('iob'))
            .clip(lower_bound=0.0)
            .alias('phys_net_need'),
        pl.lit(0.0).alias('is_synthetic')  # Tag real data
    ])

df_real = add_physics_features(df_real)
print("✅ Physics features added to real data")

In [ ]:
# v10 Feature List (20 features)
V10_FEATURES = [
    'CGM', 'carbs', 'iob',
    'phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need',
    'est_tdd', 'est_isf', 'est_icr',
    'cgm_delta_5min', 'cgm_delta_15min', 'cgm_velocity',
    'basal', 'basal_delta_5min',
    'time_sin', 'time_cos', 'is_weekend',
    'age', 'weight', 'gender_encoded'
]

# Align columns
common_cols = V10_FEATURES + ['bolus', 'is_synthetic']

# Check missing columns
missing_real = [c for c in common_cols if c not in df_real.columns]
missing_synth = [c for c in common_cols if c not in df_synthetic.columns]

if missing_real:
    print(f"⚠️ Missing in real: {missing_real}")
if missing_synth:
    print(f"⚠️ Missing in synthetic: {missing_synth}")

# Select common columns
df_real_aligned = df_real.select(common_cols)
df_synthetic_aligned = df_synthetic.select(common_cols)

print(f"\n📊 Real aligned: {df_real_aligned.shape}")
print(f"📊 Synthetic aligned: {df_synthetic_aligned.shape}")

In [ ]:
# Merge datasets
print("\n🔗 Merging datasets...")
df_combined = pl.concat([df_real_aligned, df_synthetic_aligned])

# Shuffle
df_combined = df_combined.sample(fraction=1.0, shuffle=True, seed=42)

print(f"\n✅ Combined Dataset: {len(df_combined):,} rows")
print(f"   Real: {len(df_real_aligned):,} ({100*len(df_real_aligned)/len(df_combined):.1f}%)")
print(f"   Synthetic: {len(df_synthetic_aligned):,} ({100*len(df_synthetic_aligned)/len(df_combined):.1f}%)")

# Memory cleanup
del df_real, df_real_aligned, df_synthetic, df_synthetic_aligned
gc.collect()

## Step 3: Aggressive Sample Weighting

In [ ]:
print("="*70)
print("⚖️ AGGRESSIVE SAMPLE WEIGHTING")
print("="*70)

# Extract arrays
X_train = df_combined.select(V10_FEATURES).to_numpy().astype(np.float32)
y_train = df_combined.select('bolus').to_numpy().ravel().astype(np.float32)
is_synthetic = df_combined.select('is_synthetic').to_numpy().ravel()

# Clean inf/nan
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\n📊 X_train: {X_train.shape}")
print(f"📊 y_train: {y_train.shape}")

In [ ]:
# Weighting Strategy:
# - Real data: Weight 1.0
# - Synthetic data: Weight 2.0 (counts as 4M in loss function)

sample_weights = np.where(is_synthetic == 1.0, 2.0, 1.0).astype(np.float32)

print(f"\n📊 Weight Distribution:")
print(f"   Weight 1.0 (Real): {(sample_weights == 1.0).sum():,} samples")
print(f"   Weight 2.0 (Synthetic): {(sample_weights == 2.0).sum():,} samples")
print(f"\n   Effective Real Rows: {(sample_weights == 1.0).sum():,}")
print(f"   Effective Synthetic Rows: {(sample_weights == 2.0).sum() * 2:,} (2x weight)")
print(f"   Total Effective: {sample_weights.sum():,.0f}")

# Memory cleanup
del df_combined, is_synthetic
gc.collect()

In [ ]:
# Load test data
print("\n📂 Loading test data...")
df_test = pl.read_parquet(TEST_FILE)
df_test = add_physics_features(df_test)

X_test = df_test.select(V10_FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select('bolus').to_numpy().ravel().astype(np.float32)

X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

print(f"✅ Test: {X_test.shape}")

## Step 4: Monotonic Constraints

In [ ]:
print("="*70)
print("🔒 MONOTONIC CONSTRAINTS")
print("="*70)

CONSTRAINTS = {
    'CGM': 1,
    'carbs': 1,
    'iob': -1,
    'phys_meal_bolus': 1,
    'phys_corr_bolus': 1,
    'phys_net_need': 1,
    'est_isf': -1,
    'est_icr': -1,
    'cgm_delta_5min': 1,
    'cgm_delta_15min': 1,
    'cgm_velocity': 1,
}

monotone_tuple = tuple(CONSTRAINTS.get(f, 0) for f in V10_FEATURES)

constrained = sum(1 for c in monotone_tuple if c != 0)
print(f"\n✅ Constraints active: {constrained}/{len(V10_FEATURES)} features")

## Step 5: Train XGBoost v10

In [ ]:
print("="*70)
print("🚀 TRAINING XGBOOST v10 (MASSIVE INJECTION)")
print("="*70)

params = {
    'n_estimators': 3000,
    'learning_rate': 0.02,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'monotone_constraints': monotone_tuple,
    'early_stopping_rounds': 100
}

print(f"\n📋 Parameters:")
for k, v in params.items():
    if k != 'monotone_constraints':
        print(f"   {k}: {v}")

print(f"\n📊 Training size: {len(X_train):,} rows")
print(f"📊 Effective size: {sample_weights.sum():,.0f} (with weights)")

In [ ]:
# TRAIN!
print("\n🏋️ Training with Aggressive Weighting...")
print("   (This may take 5-15 minutes on GPU)")

model = xgb.XGBRegressor(**params)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=200
)

print(f"\n✅ Training Complete!")
print(f"   Best Iteration: {model.best_iteration}")
print(f"   Best Score: {model.best_score:.6f}")

## Step 6: THE FINAL VERDICT - 4-Scenario Stress Test

In [ ]:
print("="*70)
print("🎯 THE FINAL VERDICT - 4-SCENARIO STRESS TEST")
print("="*70)

def create_test_input(cgm, carbs, iob, est_isf=30, est_icr=10, velocity=0):
    phys_meal = carbs / est_icr if est_icr > 0 else 0
    phys_corr = max(0, (cgm - TARGET_BG) / est_isf) if est_isf > 0 else 0
    phys_net = max(0, phys_meal + phys_corr - iob)
    
    return np.array([[
        cgm, carbs, iob,
        phys_meal, phys_corr, phys_net,
        40.0, est_isf, est_icr,
        velocity, velocity*3, velocity/5,
        1.0, 0.0,
        0.0, 1.0, 0.0,
        30.0, 70.0, 0.0
    ]], dtype=np.float32)

def calc_math_dose(cgm, carbs, iob, isf=30, icr=10, target=100):
    meal = carbs / icr if icr > 0 and carbs > 0 else 0
    corr = max(0, (cgm - target) / isf) if cgm > target else 0
    return max(0, meal + corr - iob)

# Define scenarios with STRICT targets
scenarios = [
    {"name": "A: Meal (Normal BG)", "cgm": 120, "carbs": 50, "iob": 0, "velocity": 0,
     "target_range": (4.5, 7.5), "description": "Meal bolus ~5-7U"},
    {"name": "B: Correction (High BG)", "cgm": 250, "carbs": 0, "iob": 0, "velocity": 2,
     "target_range": (4.5, 6.5), "description": "MUST be > 4.5U!"},
    {"name": "C: Meal + Correction", "cgm": 180, "carbs": 40, "iob": 1, "velocity": 1,
     "target_range": (4.0, 8.0), "description": "Combined bolus"},
    {"name": "D: Hypo Risk", "cgm": 80, "carbs": 0, "iob": 0, "velocity": -3,
     "target_range": (0.0, 0.5), "description": "MUST be < 0.5U!"},
]

print(f"\n{'Scenario':<30} {'Math':>8} {'v10 AI':>8} {'Target':>12} {'Status':>8}")
print("-"*70)

all_pass = True
results = []

for s in scenarios:
    X = create_test_input(s['cgm'], s['carbs'], s['iob'], velocity=s['velocity'])
    math_dose = calc_math_dose(s['cgm'], s['carbs'], s['iob'])
    ai_dose = max(0, model.predict(X)[0])
    
    # Check against target range
    low, high = s['target_range']
    in_range = low <= ai_dose <= high
    status = "✅ PASS" if in_range else "❌ FAIL"
    
    if not in_range:
        all_pass = False
    
    results.append({
        'name': s['name'],
        'math': math_dose,
        'ai': ai_dose,
        'target': s['target_range'],
        'status': status
    })
    
    target_str = f"{low:.1f}-{high:.1f}U"
    print(f"{s['name']:<30} {math_dose:>8.2f} {ai_dose:>8.2f} {target_str:>12} {status:>8}")

print("-"*70)

if all_pass:
    print("\n🎉🎉🎉 ALL SCENARIOS PASSED! 🎉🎉🎉")
    print("\n✅ Correction Bolus Blindness: CURED!")
else:
    print("\n⚠️ Some scenarios failed - check model performance")

In [ ]:
# Critical comparison: A vs B
print("\n" + "="*70)
print("🔬 CRITICAL CHECK: A vs B (Must be DIFFERENT!)")
print("="*70)

pred_A = results[0]['ai']
pred_B = results[1]['ai']

print(f"\nScenario A (Meal, 120 mg/dL, 50g carbs): {pred_A:.2f}U")
print(f"Scenario B (Correction, 250 mg/dL, 0g carbs): {pred_B:.2f}U")
print(f"\nDifference: {abs(pred_A - pred_B):.2f}U")

if abs(pred_A - pred_B) < 1.0:
    print("\n⚠️ WARNING: A and B too similar!")
else:
    print("\n✅ A and B are properly differentiated!")

## Step 7: Feature Importance

In [ ]:
print("="*70)
print("📊 v10 FEATURE IMPORTANCE")
print("="*70)

importance = model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

print("\n🏆 Top 10 Features:")
for rank, idx in enumerate(sorted_idx[:10], 1):
    feat = V10_FEATURES[idx]
    imp = importance[idx]
    print(f"   {rank:2d}. {feat:<20} {imp:.4f}")

# Check critical features
print("\n🔬 Critical Feature Rankings:")
critical = ['phys_corr_bolus', 'CGM', 'carbs', 'phys_meal_bolus', 'iob']
for feat in critical:
    idx = V10_FEATURES.index(feat)
    rank = np.where(sorted_idx == idx)[0][0] + 1
    imp = importance[idx]
    print(f"   {feat:<20} Rank #{rank} (imp={imp:.4f})")

In [ ]:
# Plot
plt.figure(figsize=(12, 8))

top_n = 15
top_features = [V10_FEATURES[i] for i in sorted_idx[:top_n]]
top_importance = importance[sorted_idx[:top_n]]

colors = ['#ff4b4b' if 'phys_' in f else '#667eea' if f in ['carbs', 'CGM', 'iob'] else '#38ef7d' 
          for f in top_features]

plt.barh(range(top_n), top_importance[::-1], color=colors[::-1])
plt.yticks(range(top_n), top_features[::-1])
plt.xlabel('Importance')
plt.title('v10 Feature Importance (After 2M Synthetic Injection)\n🔴 Physics | 🟣 Core Clinical | 🟢 Other')
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/v10_feature_importance.png", dpi=150)
plt.show()

## Step 8: Save Model

In [ ]:
print("="*70)
print("💾 SAVING v10 MODEL")
print("="*70)

os.makedirs(MODEL_PATH, exist_ok=True)

# Save model
model_file = f"{MODEL_PATH}/xgboost_clinical_v10.json"
model.save_model(model_file)
print(f"\n✅ Model: {model_file}")

# Save config
config = {
    'version': 'v10',
    'features': V10_FEATURES,
    'target': TARGET,
    'physics_features': ['phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need'],
    'monotone_constraints': list(monotone_tuple),
    'constraint_map': CONSTRAINTS,
    'target_bg': TARGET_BG,
    'synthetic_injection': SYNTHETIC_SIZE,
    'synthetic_weight': 2.0,
    'best_iteration': model.best_iteration,
    'best_score': float(model.best_score),
    'stress_test_results': results
}

config_file = f"{MODEL_PATH}/model_config_v10.json"
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"✅ Config: {config_file}")

In [ ]:
# Final metrics
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, 25)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*70)
print("🎉 v10 TRAINING COMPLETE")
print("="*70)
print(f"\n📊 Test Metrics:")
print(f"   RMSE: {rmse:.4f}")
print(f"   MAE:  {mae:.4f}")
print(f"\n🧬 Synthetic Injection: {SYNTHETIC_SIZE:,} rows (2.0x weight)")
print(f"🔒 Constrained Features: {constrained}")
print(f"\n🚀 Ready for app.py integration!")